# Auditoría para diseño de identificación — 2026-09-20

Revisión de cobertura y superposición de acreedores. No estima efectos causales. Los outputs son agregados, sin identificadores de deudores. Ejecutar desde el repositorio o una subcarpeta; requiere pandas y duckdb.

**Límites:** se revisan tres cortes para el cruce CENDEU. Personas es la muestra existente llamada 5pct; no se revalidó aquí su selección. Se exige saldo positivo sin restringir situación ni actividad. Fintech incluye solo el grupo pnfc_fintech del crosswalk actual, que requiere auditoría. El indicador bancario usa es_banco de cualquier mes del panel y aún debe fecharse. Registros sin correspondencia no se cuentan como bancos ni fintech. Los datos locales no se modifican.

In [1]:
from pathlib import Path
import json
import pandas as pd
import duckdb

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'Data/processed/balances_bancos.dta').exists())
balances = pd.read_stata(ROOT / 'Data/processed/balances_bancos.dta', convert_categoricals=False)
crosswalk = pd.read_csv(ROOT / 'Data/crosswalks/cendeu_lenders.csv')
assert crosswalk.ent.is_unique
audit = {
    'balances': {'rows': len(balances), 'columns': len(balances.columns),
                 'entities': int(balances.ent.nunique()),
                 'first_month': str(balances.ym.min().date()),
                 'last_month': str(balances.ym.max().date()),
                 'duplicate_entity_month': int(balances.duplicated(['ent','ym']).sum())}
}
digital_ids = [143, 384, 45030, 45072]
digital = balances.loc[(balances.ym == '2025-05-01') & balances.ent.isin(digital_ids), ['ent','nombre','es_banco']]
audit['regulated_digital_entities_may2025'] = digital.to_dict('records')

# Provisional bank classification: any observation with es_banco==1 in the IEF panel.
# This is not a validated historical legal-status crosswalk. Fintech is narrow PNFC only.
bank_ids = set(balances.loc[balances.es_banco.eq(1), 'ent'].astype(int))
tax = crosswalk[['ent','group']].copy()
tax['is_bank'] = tax.ent.isin(bank_ids)
tax['is_fintech'] = tax['group'].eq('pnfc_fintech')
con = duckdb.connect()
con.register('tax', tax)
sql = """
WITH person AS (
 SELECT p.cuit,
        count(DISTINCT p.ent) FILTER (WHERE coalesce(t.is_bank, false)) AS n_bank,
        bool_or(coalesce(t.is_fintech, false)) AS has_fintech
 FROM read_parquet(?) p LEFT JOIN tax t USING (ent)
 WHERE p.amount_ars > 0
 GROUP BY p.cuit
)
SELECT count(*) AS borrowers,
       count(*) FILTER (WHERE n_bank >= 2) AS multi_bank,
       count(*) FILTER (WHERE has_fintech) AS with_pnfc_fintech,
       count(*) FILTER (WHERE n_bank >= 1 AND has_fintech) AS bank_and_fintech,
       count(*) FILTER (WHERE n_bank >= 2 AND has_fintech) AS fintech_and_multi_bank
FROM person
"""
audit['overlap'] = []
for population in ['firm', 'person_lender_5pct']:
    for part in ['23', '14', '00']:
        path = ROOT / 'Data/processed/cendeu' / population / f'part-{part}.parquet'
        month = con.execute('SELECT DISTINCT year, month FROM read_parquet(?)', [str(path)]).fetchall()
        assert len(month) == 1
        row = con.execute(sql, [str(path)]).fetchdf().to_dict('records')[0]
        row.update(population=population, month=f'{month[0][0]:04}-{month[0][1]:02}')
        audit['overlap'].append(row)
audit['taxonomy_review'] = crosswalk.loc[
    crosswalk.ent.isin([44077,44088,44092,44093,44094,44095,44096,44098,44099,65203]),
    ['ent','name','group']].to_dict('records')
con.close()
print(json.dumps(audit, ensure_ascii=False, indent=2))


{
  "balances": {
    "rows": 7681,
    "columns": 328,
    "entities": 84,
    "first_month": "2018-01-01",
    "last_month": "2026-04-01",
    "duplicate_entity_month": 0
  },
  "regulated_digital_entities_may2025": [
    {
      "ent": 143.0,
      "nombre": "BRUBANK S.A.U.",
      "es_banco": 1
    },
    {
      "ent": 384.0,
      "nombre": "UALA BANK S.A.U.",
      "es_banco": 1
    },
    {
      "ent": 45030.0,
      "nombre": "NARANJA DIGITAL COMPAÑÍA FINANCIERA S.A.U.",
      "es_banco": 0
    },
    {
      "ent": 45072.0,
      "nombre": "REBA COMPAÑIA FINANCIERA S.A.",
      "es_banco": 0
    }
  ],
  "overlap": [
    {
      "borrowers": 225712,
      "multi_bank": 62917,
      "with_pnfc_fintech": 3461,
      "bank_and_fintech": 2281,
      "fintech_and_multi_bank": 909,
      "population": "firm",
      "month": "2024-08"
    },
    {
      "borrowers": 254195,
      "multi_bank": 74958,
      "with_pnfc_fintech": 5282,
      "bank_and_fintech": 3951,
      "fintech_an